In [1]:
# 0. IMPORTS
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix, precision_score, recall_score

In [2]:
# 1. LOAD THE CHOSEN MODEL AND TEST DATA FROM NOTEBOOK 05
final_lgb_model = joblib.load('/kaggle/input/datasets/hamaz911/notebook-06-threshold-optimization/lgb_notebook05_model.pkl')

X_test = pd.read_parquet('/kaggle/input/datasets/hamaz911/notebook-06-threshold-optimization/X_test.parquet')
y_test = pd.read_parquet('/kaggle/input/datasets/hamaz911/notebook-06-threshold-optimization/y_test.parquet')['isFraud']

print("Model loaded:", final_lgb_model)
print("X_test shape:", X_test.shape)

Model loaded: LGBMClassifier(random_state=42, scale_pos_weight=np.float64(27.46147358274595),
               verbose=-1)
X_test shape: (118108, 311)


In [3]:
# 2. GET PREDICTED PROBABILITIES
y_proba = final_lgb_model.predict_proba(X_test)[:, 1]

print("Probability distribution:")
print(pd.Series(y_proba).describe())

Probability distribution:
count    118108.000000
mean          0.225459
std           0.219987
min           0.000563
25%           0.071065
50%           0.141242
75%           0.297327
max           0.998119
dtype: float64


In [4]:
# 3. CHECK PRECISION/RECALL ACROSS SEVERAL THRESHOLDS
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    print(f"Threshold {threshold}: Precision={p:.3f}, Recall={r:.3f}")

Threshold 0.1: Precision=0.053, Recall=0.967
Threshold 0.2: Precision=0.085, Recall=0.918
Threshold 0.3: Precision=0.121, Recall=0.867
Threshold 0.4: Precision=0.157, Recall=0.813
Threshold 0.5: Precision=0.200, Recall=0.749
Threshold 0.6: Precision=0.258, Recall=0.670
Threshold 0.7: Precision=0.347, Recall=0.586
Threshold 0.8: Precision=0.484, Recall=0.472


In [5]:
# 5. COST-BASED THRESHOLD OPTIMIZATION
FN_COST = 149.24  # average fraud transaction amount (data-derived, Notebook 02)
FP_COST = 10       # assumed customer friction/goodwill cost (documented assumption)

threshold_costs = []

for threshold in np.arange(0.05, 0.95, 0.05):
    y_pred_t = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    
    total_cost = (fp * FP_COST) + (fn * FN_COST)
    
    threshold_costs.append({
        'threshold': round(threshold, 2),
        'fp': fp, 'fn': fn, 'tp': tp,
        'total_cost': total_cost
    })

cost_df = pd.DataFrame(threshold_costs)
print(cost_df)

best_row = cost_df.loc[cost_df['total_cost'].idxmin()]
print("\nBest threshold by cost:", best_row['threshold'], "with total cost $", best_row['total_cost'])

    threshold     fp    fn    tp  total_cost
0        0.05  96605    36  4028   971422.64
1        0.10  70078   134  3930   720778.16
2        0.15  52585   230  3834   560175.20
3        0.20  40368   333  3731   453376.92
4        0.25  31768   443  3621   383793.32
5        0.30  25703   542  3522   337918.08
6        0.35  21324   645  3419   309499.80
7        0.40  17812   758  3306   291243.92
8        0.45  14831   888  3176   280835.12
9        0.50  12167  1022  3042   274193.28
10       0.55   9858  1169  2895   273041.56
11       0.60   7847  1341  2723   278600.84
12       0.65   5991  1490  2574   282277.60
13       0.70   4489  1683  2381   296060.92
14       0.75   3109  1915  2149   316884.60
15       0.80   2048  2146  1918   340749.04
16       0.85   1156  2369  1695   365109.56
17       0.90    539  2650  1414   400876.00

Best threshold by cost: 0.55 with total cost $ 273041.56


In [6]:
# 6. TEST HOW OPTIMAL THRESHOLD CHANGES ACROSS DIFFERENT FP COST ASSUMPTIONS
fp_cost_scenarios = [5, 10, 20, 30, 50]

sensitivity_results = []

for fp_cost in fp_cost_scenarios:
    best_cost = float('inf')
    best_threshold = None
    
    for threshold in np.arange(0.05, 0.95, 0.05):
        y_pred_t = (y_proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
        total_cost = (fp * fp_cost) + (fn * FN_COST)
        
        if total_cost < best_cost:
            best_cost = total_cost
            best_threshold = round(threshold, 2)
    
    sensitivity_results.append({'fp_cost': fp_cost, 'best_threshold': best_threshold, 'best_total_cost': round(best_cost, 2)})

sensitivity_df = pd.DataFrame(sensitivity_results)
print(sensitivity_df)

   fp_cost  best_threshold  best_total_cost
0        5            0.40        202183.92
1       10            0.55        273041.56
2       20            0.70        340950.92
3       30            0.75        379064.60
4       50            0.85        411349.56


In [8]:
# 7. SAVE FINAL MODEL + THRESHOLD CONFIGURATION FOR THE STREAMLIT APP
import json

final_config = {
    'model_type': 'LightGBM',
    'scale_pos_weight': 27.46147358274595,
    'default_threshold': 0.55,
    'fn_cost_assumption': 149.24,
    'fp_cost_assumption': 10,
    'threshold_sensitivity': {
        'fp_cost_5': 0.40,
        'fp_cost_10': 0.55,
        'fp_cost_20': 0.70,
        'fp_cost_30': 0.75,
        'fp_cost_50': 0.85
    },
    'test_set_performance_at_default_threshold': {
        'false_positives': 9858,
        'false_negatives': 1169,
        'true_positives': 2895,
        'total_cost': 273041.56
    }
}

with open('/kaggle/working/final_model_config.json', 'w') as f:
    json.dump(final_config, f, indent=2)

# Save the final model itself (same one from Notebook 05, now with its chosen threshold documented)
joblib.dump(final_lgb_model, '/kaggle/working/final_fraud_model.pkl')

print("Saved final model and configuration.")

Saved final model and configuration.
